# Hypothesis Testing and Feature Selection

This notebook evaluates which features provide the strongest statistical and predictive value for Titanic survival prediction. The goal is to reduce noise, remove redundant variables, and create cleaner datasets for downstream model training.

Two approaches are used in this phase:

1. **Sequential Feature Selection** for K-Nearest Neighbors, since KNN is a non-parametric model and does not provide coefficient p-values.
2. **Logistic Regression / Logit Testing** to evaluate statistical significance, coefficient behavior, and feature redundancy.

The final output of this notebook is a set of curated feature datasets designed for linear and non-linear modeling.

In [ ]:
# new dataset with updated features 
from pathlib import Path  
import pandas as pd 

# instantiation process 
dir = Path.cwd()

# find the data file 
data_path = dir.parent / 'Data' / 'significant_feat.csv'

# load data in object 
df = pd.read_csv(data_path)

# view result 
df.head()


In [ ]:
# training phase 
# Peforming hyptothesis testing on a training set is key in identify which features are significant 
from sklearn.model_selection import train_test_split

# inputs/features (drop target feature to avoid data leakage)
X = df.drop(columns=['Survived'])

# Target variable (Save target variable into data as an csv for future downstream work)
Y = df['Survived']

# define where you want it to go and save it 
save_path = Path.cwd().parent / 'Data' / 'target.csv'
Y.to_csv(save_path, index = False)

# 70/30 split 
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size= 0.3, shuffle= True, random_state= 42)

#Reset indexs 
X_train = X_train.reset_index(drop = True)
Y_train = Y_train.reset_index(drop = True)

# make data 1D (data will load with index)
Y_train = Y_train.values.ravel()


# K-Nearest Neighbors Feature Selection

K-Nearest Neighbors does not produce coefficients or p-values because it is a non-parametric model. Instead of using traditional hypothesis testing, sequential feature selection was applied to identify the strongest feature subset.

Backward sequential feature selection was used to remove weaker predictors while optimizing for precision. This helped identify which features contributed most effectively to KNN performance without relying on statistical coefficient testing.


In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector # elimination process
from sklearn.neighbors import KNeighborsClassifier # model 

# initialze model with the gridsearch adjusted parameters 
knn = KNeighborsClassifier(leaf_size=10, weights='distance')

# backward elimination for KKN 
sfs = SequentialFeatureSelector(knn, 
                                n_features_to_select = 'auto', 
                                direction = 'backward', 
                                scoring = 'precision')
# fit data onto model 
sfs.fit(X_train, Y_train)

# best features
selected_features = X_train.columns[sfs.get_support()]

# results
selected_features

# Logistic Regression Significance Testing

Logistic regression was used to evaluate the statistical significance of the available features. Unlike KNN, logistic regression produces coefficients, standard errors, z-scores, and p-values, which can be used to determine whether each predictor has a meaningful relationship with survival.

Before fitting the model, categorical redundancy was reduced by removing duplicate baseline categories. This helps prevent multicollinearity and allows the model to interpret one category as the reference group.

In [ ]:
import statsmodels.api as sm 

#convert bool objects to floats (statsmodel has a hard time seeing them as bool objects)
X_train_numeric = X_train.astype(float)
                    
# manually add a constant intercept to the data 
X_train_const = sm.add_constant(X_train_numeric)

# fit the model
model = sm.Logit(Y_train, X_train_const)
result = model.fit(maxiter = 1000)

# view results 
print(f'Summary {result.summary()}')

# Variance Inflation Factor Review

Variance Inflation Factor was used to evaluate multicollinearity between input features. High VIF values indicate that a feature can be strongly explained by other features, which can make coefficient estimates unstable.

Several variables produced extremely high or infinite VIF values, suggesting strong redundancy between engineered interaction features and their original variables. These features were flagged for removal before fitting the final logistic regression model.

In [ ]:
# variance inflation error
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ensure X has a constant intercept 
X = X_train_const.copy()

# calculate VIFs
vifs = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

# create table 
pd.DataFrame({'feature' : X.columns, 'VIF': vifs})


# Final Logistic Regression Significance Test

After reviewing p-values and multicollinearity, redundant and statistically weak features were removed from the training dataset. The logistic regression model was then refit using the reduced feature set.

The final model successfully converged after feature removal, indicating that the simplified dataset was more stable for logistic regression. This step helped identify a cleaner set of predictors for the final linear modeling dataset.



In [ ]:
# remove redundant columns 
X_train_const = X_train_const.drop(columns = ['Fare', 'interaction4', 'interaction5', 'interaction6', 'interaction7', 
                                              'Embarked_644', 'Embarked_C', 'Ratio4', 'Ratio5', 'Ratio1', 'Parch', 'Embarked_Q'])

# begin model fitting 
model = sm.Logit(Y_train, X_train_const)
results = model.fit(maxiter = 2000)

# results
print(f'Summary {results.summary()}')

# Final Dataset Creation

After completing feature selection and significance testing, two curated datasets were created for downstream modeling.

The **linear dataset** was designed for Logistic Regression and includes features that are easier to interpret and less affected by redundancy.

The **non-linear dataset** was designed for models such as KNN and SVM, which can benefit from engineered interaction features and more complex relationships.

Both datasets were saved as CSV files for use in the modeling and evaluation notebooks.


In [ ]:
# Drop the uncessary features needed to improve the KNN Model 
# This will also be considered the non-linear dataset
non_linear_df = df[['Pclass', 'Age', 'Parch', 'Fare', 'Embarked_C', 'interaction4',
       'interaction5', 'interaction7', 'Ratio4']]

# drop the uncessary features needed to improve the logistic Regression Model
# This will also be considered the linear dataset 
linear_df = df[['Pclass', 'Age', 'SibSp', 'Sex_female', 'Cabin_encoded']]

# save datasets to data directory
save_path = dir.parent / 'Data' / 'linear.csv'
save_path1 = dir.parent / 'Data' / 'non_linear.csv'

# linear csv
linear_df.to_csv(save_path, index = False)
# non_linear.csv 
non_linear_df.to_csv(save_path1, index = False)

